# Chess Analytics MVP — Python EDA Foundation

This notebook profiles the five completed 100,000-game Parquet batches (500,000 games). They are an early-August sample from the August archive, not a representative monthly sample. DuckDB SQL remains the detailed analysis layer; this notebook prepares reproducible Python EDA for a later Power BI dashboard. All results are observational, not causal.

## Safe access

Only analytical columns are selected. Player identifiers are deliberately excluded from the view and never displayed.

In [ ]:
from pathlib import Path
import os
import duckdb
import pandas as pd

start = Path.cwd().resolve()
PROJECT_ROOT = start if (start / 'data').exists() else start.parent
if not (PROJECT_ROOT / 'data').exists():
    raise RuntimeError('Run from the project root or notebooks directory.')
os.environ.setdefault('MPLCONFIGDIR', str(PROJECT_ROOT / 'data' / 'processed' / '.matplotlib_cache'))
DATA_DIR = PROJECT_ROOT / 'data' / 'processed' / 'mvp_500k_batches'
files = sorted(DATA_DIR.glob('games_*.parquet'))
assert len(files) == 5, f'Expected five completed batches, found {len(files)}'
con = duckdb.connect()
parquet_glob = str(DATA_DIR / 'games_*.parquet').replace("'", "''")
con.execute(f"""CREATE OR REPLACE TEMP VIEW games AS
SELECT game_id, utc_timestamp, white_elo, black_elo, white_rating_diff,
       black_rating_diff, result, eco, opening, time_control, speed_category, termination, event
FROM read_parquet('{parquet_glob}')
""")
row_count = con.execute('SELECT COUNT(*) FROM games').fetchone()[0]
assert row_count == 500_000, f'Expected 500,000 rows, found {row_count}'
print(f'Loaded {len(files)} completed batches and {row_count:,} games.')

## Schema and data quality

These compact summaries use DuckDB aggregation; only small result tables enter pandas.

In [ ]:
schema = con.execute('DESCRIBE games').df()[['column_name', 'column_type', 'null']]
schema

quality = con.execute("""SELECT
  COUNT(*) AS total_games,
  COUNT(*) - COUNT(DISTINCT game_id) AS duplicate_game_ids,
  SUM(white_elo IS NULL OR black_elo IS NULL OR white_elo NOT BETWEEN 100 AND 4000 OR black_elo NOT BETWEEN 100 AND 4000) AS invalid_ratings,
  SUM(result IS NULL OR result NOT IN ('1-0','0-1','1/2-1/2')) AS invalid_results,
  SUM(opening IS NULL OR opening IN ('','?')) AS missing_or_unknown_openings,
  SUM(white_rating_diff IS NOT NULL AND black_rating_diff IS NOT NULL) AS rating_change_pairs
FROM games
"").df()
quality

con.execute("""SELECT speed_category, time_control, COUNT(*) AS games
FROM games GROUP BY 1,2 ORDER BY games DESC LIMIT 20"").df()

## Feature engineering

Definitions exactly match the validated SQL: ratings are valid only from 100–4000; upset means the lower-rated player wins with a gap of at least 100; comparable opponents have a gap no greater than 100.

In [ ]:
con.execute("""CREATE OR REPLACE TEMP VIEW featured_games AS
SELECT *,
 white_elo BETWEEN 100 AND 4000 AND black_elo BETWEEN 100 AND 4000 AS has_valid_ratings,
 result IN ('1-0','0-1','1/2-1/2') AS has_valid_result,
 ABS(white_elo-black_elo) AS rating_gap,
 CASE WHEN white_elo NOT BETWEEN 100 AND 4000 OR black_elo NOT BETWEEN 100 AND 4000 THEN NULL
      WHEN ABS(white_elo-black_elo)<50 THEN '0-49 Closely matched'
      WHEN ABS(white_elo-black_elo)<100 THEN '50-99 Small difference'
      WHEN ABS(white_elo-black_elo)<200 THEN '100-199 Moderate difference'
      WHEN ABS(white_elo-black_elo)<400 THEN '200-399 Large difference'
      ELSE '400+ Very large difference' END AS rating_gap_category,
 CASE WHEN white_elo NOT BETWEEN 100 AND 4000 THEN NULL WHEN white_elo<800 THEN 'Below 800' WHEN white_elo<1200 THEN '800-1199' WHEN white_elo<1500 THEN '1200-1499' WHEN white_elo<1800 THEN '1500-1799' WHEN white_elo<2100 THEN '1800-2099' ELSE '2100+' END AS white_rating_band,
 CASE WHEN black_elo NOT BETWEEN 100 AND 4000 THEN NULL WHEN black_elo<800 THEN 'Below 800' WHEN black_elo<1200 THEN '800-1199' WHEN black_elo<1500 THEN '1200-1499' WHEN black_elo<1800 THEN '1500-1799' WHEN black_elo<2100 THEN '1800-2099' ELSE '2100+' END AS black_rating_band,
 CASE WHEN white_elo<black_elo AND result='1-0' OR black_elo<white_elo AND result='0-1' THEN TRUE ELSE FALSE END AS lower_rated_player_won,
 has_valid_ratings AND has_valid_result AND ABS(white_elo-black_elo)>=100 AS upset_eligible,
 has_valid_ratings AND has_valid_result AND ABS(white_elo-black_elo)>=100 AND (white_elo<black_elo AND result='1-0' OR black_elo<white_elo AND result='0-1') AS is_upset,
 has_valid_ratings AND has_valid_result AND ABS(white_elo-black_elo)<=100 AS comparable_opponents
FROM games
"")
con.execute('SELECT rating_gap_category, COUNT(*) AS games FROM featured_games WHERE has_valid_ratings GROUP BY 1 ORDER BY MIN(rating_gap)').df()

## Initial sanity checks

Assertions validate feature boundaries without exposing player-level data.

In [ ]:
checks = con.execute("""SELECT
 COUNT(*) AS featured_rows,
 SUM(is_upset) AS upsets, SUM(upset_eligible) AS upset_eligible,
 SUM(is_upset AND (result='1/2-1/2' OR white_elo=black_elo)) AS invalid_upsets,
 SUM(has_valid_ratings AND rating_gap_category IS NULL) AS uncategorized_valid_ratings,
 SUM((white_elo NOT BETWEEN 100 AND 4000) AND white_rating_band IS NOT NULL) AS invalid_white_band,
 SUM((black_elo NOT BETWEEN 100 AND 4000) AND black_rating_band IS NOT NULL) AS invalid_black_band
FROM featured_games
"").fetchone()
assert checks[0] == 500_000
assert checks[1] <= checks[2]
assert checks[3] == 0
assert checks[4] == checks[5] == checks[6] == 0
assert 'white_player_id' not in [r[0] for r in con.execute('DESCRIBE featured_games').fetchall()]
print('All Stage 1 sanity checks passed.')
pd.DataFrame([checks], columns=['featured_rows','upsets','upset_eligible','invalid_upsets','uncategorized_valid_ratings','invalid_white_band','invalid_black_band'])